In [4]:
import sys
import xarray as xr
import numpy as np
from matplotlib import pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature
import geopandas as gpd
from shapely.geometry import mapping
from scipy.stats import spearmanr, pearsonr

import warnings
warnings.filterwarnings('ignore')

In [5]:
datap = "/Users/ellendyer/Documents/GitHub/Isotopes_F4R/plots/"
dataf = "/Users/ellendyer/Documents/GitHub/F4R_data/"

### Read in ERA5 2m temperature 
- do this for whole available ts and then sub-select years in next step for analysis

In [6]:
Y1=2018
Y2=2024

t2m_all_list = []
for Y in range(Y1,Y2+1):
    
    t2m = xr.open_mfdataset('/Volumes/New_5TB/ESA_F4R/ERA5_temp/t2m_'+str(Y)+'.nc')['t2m']
    t2m = t2m.rename({'latitude':'lat','longitude':'lon','valid_time':'time'})
    t2m = t2m.sortby('lat')
    t2m = t2m.sel(lat=slice(-15,12),lon=slice(8,31),drop=True).load()
    t2m = t2m.interp(lat=np.arange(t2m["lat"].min().values,t2m["lat"].max().values,0.25), lon=np.arange(t2m["lon"].min().values,t2m["lon"].max().values,0.25), method="linear")
    t2m_year_list = []
    for m in range(1,13):
        #try:
        mp = t2m.sel(time=(t2m.time.dt.month==m), drop=True)
        bins = [mp.time[0].values,mp.time[10-1].values,mp.time[20-1].values,mp.time[-1].values]
        mp_out = mp.groupby_bins('time', bins,labels=[mp.time[10-1].values,mp.time[20-1].values,mp.time[-1].values]).mean()
        mp_out = mp_out.rename({'time_bins':'time'})
        #print(mp_out)
        t2m_year_list.append(mp_out)
        #except:
        #    print('no month - ',m,' for year - ',Y)
    t2m_year = xr.concat(t2m_year_list,dim='time')
    t2m_all_list.append(t2m_year)
    t2m.close()
    print('done - ',Y)
t2m_all = xr.concat(t2m_all_list,dim='time')
t2m_all = t2m_all.sel(time=slice('2018-07-01','2024-12-31'))
t2m_all = t2m_all.drop('number')
print(t2m_all)
    
t2m_all.to_netcdf(dataf+'era5_10day_reg_regrid.nc',engine='h5netcdf')
        

done -  2018
done -  2019
done -  2020
done -  2021
done -  2022
done -  2023
done -  2024
<xarray.DataArray 't2m' (time: 234, lat: 108, lon: 92)> Size: 19MB
array([[[292.90015327, 292.73945448, 292.61836073, ..., 292.36016846,
         292.02797445, 290.69274224],
        [292.97046577, 292.76815457, 292.64396837, ..., 292.42364502,
         292.21026611, 291.44312202],
        [293.18596056, 292.99618191, 292.86635335, ..., 292.27341715,
         292.67662896, 292.77195231],
        ...,
        [297.45348443, 297.94241672, 298.34665934, ..., 298.99178738,
         298.28768582, 298.44154867],
        [298.09579129, 298.59676785, 298.99553087, ..., 298.83358426,
         298.66897922, 298.81844754],
        [298.62535943, 298.97214762, 299.32615153, ..., 299.3717787 ,
         299.2444458 , 298.86543104]],

       [[291.70887451, 291.50477295, 291.30379639, ..., 292.2493042 ,
         291.86331787, 290.46595459],
        [291.81033936, 291.6305542 , 291.45506592, ..., 292.20467529,
 